# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.004831,0.002142,0.993027,0.005220,0.002128,0.992652,0.005655,0.000463,0.993882
1,0.998229,0.000143,0.001628,0.997774,0.000322,0.001904,0.994638,0.000368,0.004994
2,0.003982,0.000584,0.995434,0.002362,0.000710,0.996928,0.002327,0.000919,0.996754
3,0.007424,0.001954,0.990622,0.002922,0.003628,0.993450,0.003433,0.002374,0.994194
4,0.993583,0.000012,0.006405,0.997151,0.000013,0.002836,0.994112,0.000005,0.005883


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.011986,0.002516,0.985498,0.009749,0.003956,0.986295,0.008185,0.005526,0.986288
1,0.469590,0.002922,0.527488,0.344531,0.002104,0.653365,0.530403,0.002094,0.467503
2,0.998492,0.000991,0.000517,0.998555,0.001132,0.000313,0.996306,0.003216,0.000478
3,0.986619,0.000009,0.013372,0.978274,0.000024,0.021702,0.990683,0.000093,0.009224
4,0.008520,0.001975,0.989505,0.005601,0.002080,0.992319,0.003507,0.001947,0.994546


# Machine Learning

In [6]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [7]:
cols_use = ['cat_0', 'cat_1', 'cat_2']

In [8]:
def objective(trial, X, y):

    w0 = trial.suggest_float('weight_class_0', 0.0, 10.0, log=False)
    w1 = trial.suggest_float('weight_class_1', 0.0, 10.0, log=False)
    w2 = trial.suggest_float('weight_class_2', 0.0, 10.0, log=False)
    weights = np.array([w0, w1, w2])

    proba = X.loc[:, cols_use].to_numpy()
    
    weighted_probas = proba * weights
    pred = np.argmax(weighted_probas, axis=1)
    
    score = balanced_accuracy_score(y, pred)

    return score


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=2000, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-03 09:50:14,350] A new study created in memory with name: no-name-5d0c3f88-c5d6-4a11-9e3f-98920624c122
Best trial: 14. Best value: 0.948807:   1%|▊                                                                                                                                     | 12/2000 [00:00<00:53, 37.37it/s]

[I 2026-07-03 09:50:14,511] Trial 0 finished with value: 0.9081899080916727 and parameters: {'weight_class_0': 1.8504156676340033, 'weight_class_1': 4.839804389213194, 'weight_class_2': 2.1793461356423007}. Best is trial 0 with value: 0.9081899080916727.
[I 2026-07-03 09:50:14,512] Trial 4 finished with value: 0.9072189148016069 and parameters: {'weight_class_0': 4.239425971617371, 'weight_class_1': 6.681493076506772, 'weight_class_2': 6.920529492262113}. Best is trial 0 with value: 0.9081899080916727.
[I 2026-07-03 09:50:14,522] Trial 9 finished with value: 0.8265357480135945 and parameters: {'weight_class_0': 8.738591958520324, 'weight_class_1': 1.6218490964789622, 'weight_class_2': 0.9750934524880073}. Best is trial 0 with value: 0.9081899080916727.
[I 2026-07-03 09:50:14,524] Trial 8 finished with value: 0.8668144939154018 and parameters: {'weight_class_0': 5.026024678075837, 'weight_class_1': 2.5321355809708876, 'weight_class_2': 5.510814737570867}. Best is trial 0 with value: 0.9

Best trial: 16. Best value: 0.948868:   1%|█▋                                                                                                                                    | 25/2000 [00:00<00:41, 48.09it/s]

[I 2026-07-03 09:50:14,682] Trial 12 finished with value: 0.9151376900348832 and parameters: {'weight_class_0': 3.430228504058569, 'weight_class_1': 9.914882406232275, 'weight_class_2': 4.725921527005453}. Best is trial 2 with value: 0.9399934348826756.
[I 2026-07-03 09:50:14,688] Trial 14 finished with value: 0.9488068006175592 and parameters: {'weight_class_0': 0.5265510338705243, 'weight_class_1': 9.493811346999472, 'weight_class_2': 9.74596568305854}. Best is trial 14 with value: 0.9488068006175592.
[I 2026-07-03 09:50:14,719] Trial 15 finished with value: 0.9480837351633206 and parameters: {'weight_class_0': 0.3224889263709626, 'weight_class_1': 7.23777684958238, 'weight_class_2': 9.157969101987575}. Best is trial 14 with value: 0.9488068006175592.
[I 2026-07-03 09:50:14,726] Trial 16 finished with value: 0.9488679748629171 and parameters: {'weight_class_0': 0.5414227907699107, 'weight_class_1': 7.651353337473742, 'weight_class_2': 9.650381632019668}. Best is trial 16 with value: 

Best trial: 16. Best value: 0.948868:   2%|██▍                                                                                                                                   | 37/2000 [00:00<00:37, 51.98it/s]

...

[I 2026-07-03 09:51:01,133] Trial 1977 finished with value: 0.9488687783429107 and parameters: {'weight_class_0': 0.4274257999713698, 'weight_class_1': 9.401220485814887, 'weight_class_2': 7.062320802926184}. Best is trial 1025 with value: 0.9492489287165894.
[I 2026-07-03 09:51:01,149] Trial 1978 finished with value: 0.94894739387312 and parameters: {'weight_class_0': 0.45000621574519556, 'weight_class_1': 9.399151320405064, 'weight_class_2': 7.020483396442291}. Best is trial 1025 with value: 0.9492489287165894.
[I 2026-07-03 09:51:01,174] Trial 1979 finished with value: 0.9485894608088395 and parameters: {'weight_class_0': 0.4142637573566589, 'weight_class_1': 9.349271688760037, 'weight_class_2': 8.221272882376983}. Best is trial 1025 with value: 0.9492489287165894.
[I 2026-07-03 09:51:01,195] Trial 1980 finished with value: 0.9490263985875801 and parameters: {'weight_class_0': 0.8120959302637645, 'weight_class_1': 9.41232174702775, 'weight_class_2': 8.19308508875838}. Best is trial 

Best trial: 1025. Best value: 0.949249: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍| 1992/2000 [00:47<00:00, 34.50it/s]

[I 2026-07-03 09:51:01,317] Trial 1983 finished with value: 0.9489608215987727 and parameters: {'weight_class_0': 0.8169344326564585, 'weight_class_1': 9.720886583406958, 'weight_class_2': 7.107949867060314}. Best is trial 1025 with value: 0.9492489287165894.
[I 2026-07-03 09:51:01,360] Trial 1986 finished with value: 0.9491490513589915 and parameters: {'weight_class_0': 0.8519891112200692, 'weight_class_1': 9.776036758162627, 'weight_class_2': 9.115853439479485}. Best is trial 1025 with value: 0.9492489287165894.
[I 2026-07-03 09:51:01,378] Trial 1985 finished with value: 0.9411316150046783 and parameters: {'weight_class_0': 2.3386826340329665, 'weight_class_1': 9.668719166997414, 'weight_class_2': 7.6453097724935475}. Best is trial 1025 with value: 0.9492489287165894.
[I 2026-07-03 09:51:01,405] Trial 1987 finished with value: 0.946692113571487 and parameters: {'weight_class_0': 0.8245896799450605, 'weight_class_1': 9.705946195257487, 'weight_class_2': 3.5492254499326874}. Best is tr

Best trial: 1025. Best value: 0.949249: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:47<00:00, 42.28it/s]

[I 2026-07-03 09:51:01,559] Trial 1993 finished with value: 0.6602955825075755 and parameters: {'weight_class_0': 0.007630279751391855, 'weight_class_1': 9.738375876204852, 'weight_class_2': 7.652295979467857}. Best is trial 1025 with value: 0.9492489287165894.
[I 2026-07-03 09:51:01,586] Trial 1994 finished with value: 0.9433880173742084 and parameters: {'weight_class_0': 0.1404206252751975, 'weight_class_1': 9.684358611347816, 'weight_class_2': 7.659394759281316}. Best is trial 1025 with value: 0.9492489287165894.
[I 2026-07-03 09:51:01,604] Trial 1995 finished with value: 0.6574914921550468 and parameters: {'weight_class_0': 0.0029149465278841546, 'weight_class_1': 9.749136617540099, 'weight_class_2': 7.388160450796173}. Best is trial 1025 with value: 0.9492489287165894.
[I 2026-07-03 09:51:01,618] Trial 1996 finished with value: 0.9491656116182323 and parameters: {'weight_class_0': 0.6198264568103994, 'weight_class_1': 9.71906082839438, 'weight_class_2': 7.375062469725183}. Best is

In [9]:
weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = X_test.loc[:, cols_use].to_numpy() * weights

pred = np.argmax(weighted_probas, axis=1)

In [10]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [11]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.2.0_catboost_submission.csv', index=False)

In [12]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
